In [ ]:
import folium
from folium.plugins import Draw
from IPython.display import display
import json
from ipywidgets import HTML, Output, VBox

# Create an output widget to display coordinates
out = Output()

# Create a folium map centered at a given location
map_center = [55.6761, 12.5683]  # Copenhagen
my_map = folium.Map(location=map_center, zoom_start=12)

# Add drawing tools
draw = Draw(export=True, edit_options={'poly': {'allowIntersection': False}})
draw.add_to(my_map)

# Function to display map
def show_map():
    display(my_map)

# Function to capture drawn coordinates
def handle_draw(event, feature_group):
    with out:
        out.clear_output()
        coords = event['geometry']['coordinates'][0]
        print("Drawn Coordinates:", json.dumps(coords, indent=2))

# Add event listener to capture drawn features
my_map.add_child(folium.LatLngPopup())

# Display the map and output widget
show_map()
display(out)


In [ ]:
import requests
import json
import pyproj

# Define API endpoint and token
API_URL = "https://api.dataforsyningen.dk/rest/skraafoto_api/v1.0/search"
TOKEN = "ed6411d316718191439a87b042be1c4b"  # Replace with your actual token

# Define bounding box coordinates in WGS84 (lat, lon)
wgs84_bbox = [
    (55.8683, 9.8304),  # Bottom-left (lat, lon)
    (55.8723, 9.8418)   # Top-right (lat, lon)
]

# Convert WGS84 to UTM (EPSG:25832)
wgs84 = pyproj.CRS("EPSG:4326")
utm32n = pyproj.CRS("EPSG:25832")
transformer = pyproj.Transformer.from_crs(wgs84, utm32n, always_xy=True)

utm_bbox = [
    transformer.transform(wgs84_bbox[0][1], wgs84_bbox[0][0]),  # Bottom-left (Easting, Northing)
    transformer.transform(wgs84_bbox[1][1], wgs84_bbox[1][0])   # Top-right (Easting, Northing)
]





In [ ]:
# Flatten bbox for API request
bbox = [utm_bbox[0][0], utm_bbox[0][1], utm_bbox[1][0], utm_bbox[1][1]]

# API request parameters
params = {
    "bbox": ",".join(map(str, bbox)),
    "bbox-crs": "http://www.opengis.net/def/crs/EPSG/0/25832",
    "limit": 10  # Number of results to retrieve
}

# API headers
headers = {
    "token": TOKEN,
    "Content-Type": "application/geo+json"
}

# Send API request
response = requests.get(API_URL, params=params, headers=headers)

# Check for successful response
if response.status_code == 200:
    data = response.json()
    print(json.dumps(data, indent=2))  # Pretty print response
else:
    print(f"Error: {response.status_code}, {response.text}")
